# A naive Bayes classifier by hand

In [Bayes' theorem, one word at a time](one_word.ipynb), the theorem turned the presence of a word into a probability of spam. A message has many words, and the classifier should use all of them. Three problems stand in the way, and each has a simple answer: the naive assumption, smoothing, and logarithms. Together they make a naive Bayes classifier, written here in a few lines of Python and then checked against scikit-learn's.

## The messages, without repetitions

In [ ]:
from sms_spam import load_sms_spam

messages = load_sms_spam()
messages.duplicated(subset="message").sum()

np.int64(403)

403 rows repeat a message that appeared earlier (281 different messages appear more than once); "Sorry, I'll call later" alone appears 30 times. Split at random, copies of the same message would land in both the training and the test set, and the test would partly measure whether the classifier remembers messages it has already seen. Each message is kept once, and the split keeps the share of spam in both sets ([why both matter](../../ml-fundamentals/overfitting_and_the_split.ipynb)):

In [ ]:
from sklearn.model_selection import train_test_split

unique_messages = messages.drop_duplicates(subset="message")
train, test = train_test_split(unique_messages, test_size=0.2, stratify=unique_messages["label"], random_state=42)
len(train), len(test), train["label"].eq("spam").mean().round(3)

(4136, 1035, np.float64(0.126))

## Two words together

With two words, Bayes' theorem needs the probability of seeing *both* in each class, $P(\text{free}, \text{call} \mid \text{spam})$. It can be counted, and compared with the product of the two separate probabilities:

In [ ]:
import re

import pandas as pd


def words_in(message):
    return re.findall(r"[a-z0-9£]+", message.lower())


word_sets = train["message"].map(lambda message: set(words_in(message)))
is_spam = train["label"] == "spam"


def joint_and_product(first, second, class_rows):
    has_first = word_sets[class_rows].map(lambda words: first in words)
    has_second = word_sets[class_rows].map(lambda words: second in words)
    return {"P(both | class)": (has_first & has_second).mean(), "P(first | class) × P(second | class)": has_first.mean() * has_second.mean()}


pd.DataFrame({
    ("free", "call", "spam"): joint_and_product("free", "call", is_spam),
    ("free", "call", "ham"): joint_and_product("free", "call", ~is_spam),
    ("prize", "claim", "spam"): joint_and_product("prize", "claim", is_spam),
}).T.round(4)

P(both | class)  P(first | class) × P(second | class)
free  call  spam           0.0766                                0.0962
            ham            0.0025                                0.0004
prize claim spam           0.0594                                0.0144

The products are not the joint probabilities. In spam, "prize" and "claim" appear together in about 6% of the messages, four times more often than if they were independent: a message that offers a prize usually asks you to claim it. "free" and "call" appear together a little less often than independence predicts in spam, and six times more often in ham. The words of a message depend on one another.

So why not always count the combinations? Because with more words they stop existing. A message of 20 words is a combination that appears once in the data, if at all, and a new message's combination has never appeared. No amount of data can give the probability of every combination of words.

The **naive assumption** gives up on combinations: it treats the words as independent of each other *within each class*, so that the probability of a message is the product of the probabilities of its words:

$$P(w_1, w_2, \dots, w_n \mid \text{spam}) \approx P(w_1 \mid \text{spam}) \, P(w_2 \mid \text{spam}) \cdots P(w_n \mid \text{spam})$$

It is wrong, as the table shows. But the classifier doesn't need the probabilities to be right, only to be larger for the right class; the error of the assumption often pushes both classes in the same direction, and the comparison survives it.

## Counting words in each class

The model counts word *occurrences*, not just presence: a message is a bag of words, each with the number of times it appears. For each class, the probability of a word is its share of all the words in that class, which is not the probability of [Bayes' theorem, one word at a time](one_word.ipynb), the share of the messages that contain the word: counting presence leads to scikit-learn's `BernoulliNB`, counting occurrences to `MultinomialNB`.

$$P(w \mid \text{spam}) = \frac{\text{occurrences of } w \text{ in spam}}{\text{all word occurrences in spam}}$$

The counts, from the training messages only:

In [ ]:
from collections import Counter

word_counts = {
    label: Counter(word for message in train.loc[train["label"] == label, "message"] for word in words_in(message))
    for label in ("ham", "spam")
}
vocabulary = set(word_counts["ham"]) | set(word_counts["spam"])
{label: sum(counts.values()) for label, counts in word_counts.items()}, len(vocabulary)

({'ham': 53070, 'spam': 13213}, 7734)

In [ ]:
pd.DataFrame({label: [word for word, _ in counts.most_common(12)] for label, counts in word_counts.items()})

,ham,spam
0,i,to
1,you,a
2,to,call
3,the,you
4,a,your
5,u,for
6,and,free
7,in,the
8,me,2
9,it,now


The training ham holds about 53,000 word occurrences and the spam about 13,000, from a vocabulary of 7,734 different words. The most frequent words of both classes are short, common ones ("to", "you", "a"), and the lists only differ further down, where "call", "free" and "now" appear in spam. What decides the classification is not how frequent a word is, but how *differently* frequent it is in the two classes: a word as common in ham as in spam adds about the same to both scores, and says nothing.

## Words a class has never seen

A word that never appears in the training spam gets a probability of 0 in spam, and then the product for spam is 0, whatever the other words say. One unusual word in a blatant spam message would make it certain ham. The fix is **smoothing**: pretend that every word of the vocabulary was seen $\alpha$ more times in each class. With $\alpha = 1$ it is called Laplace smoothing:

$$P(w \mid \text{spam}) = \frac{\text{occurrences of } w \text{ in spam} + \alpha}{\text{all word occurrences in spam} + \alpha \, |V|}$$

where $|V|$ is the number of words in the vocabulary, so that the probabilities still add up to 1. "lol", never seen in spam, before and after:

In [ ]:
def word_probability(word, label, alpha):
    total = sum(word_counts[label].values())
    return (word_counts[label][word] + alpha) / (total + alpha * len(vocabulary))


pd.DataFrame({alpha: {label: word_probability("lol", label, alpha) for label in ("ham", "spam")} for alpha in (0, 1)}).rename_axis(columns="alpha")

alpha,0,1
ham,0.001168,0.001036
spam,0.000000,0.000048


Without smoothing, "lol" has a probability of 0 in spam, and a single "lol" would make any message ham. With $\alpha = 1$ it gets a small probability in spam, about 0.00005, some twenty times less than in ham: still strong evidence for ham, but no longer a veto. Smoothing moves probability from the frequent words to the rare ones and to those never seen: in ham, "lol" goes from 0.00117 to 0.00104.

## Products of small numbers

Each word probability is small, around one in a thousand or less, and a message multiplies dozens of them. Floating-point numbers can't go below about $10^{-308}$; beyond that, a product becomes exactly 0:

In [ ]:
import numpy as np

probabilities = np.full(120, 1e-3)
np.prod(probabilities), np.log(probabilities).sum()

(np.float64(0.0), np.float64(-828.9306334778564))

A product of 120 probabilities of one in a thousand is $10^{-360}$, stored as 0. Its logarithm is a perfectly ordinary number, about -829. Since the logarithm turns products into sums and keeps the order (the larger product has the larger logarithm), the classifier compares sums of logarithms instead:

$$\text{score}(\text{spam}) = \log P(\text{spam}) + \sum_{w \in \text{message}} \text{count}(w) \, \log P(w \mid \text{spam})$$

and predicts the class with the higher score. The two scores can be turned back into probabilities that add up to 1, without ever computing the tiny products: $P(\text{spam} \mid \text{message}) = 1 / (1 + e^{\,\text{score}(\text{ham}) - \text{score}(\text{spam})})$. That is the sigmoid of [Logistic regression](../../algorithms/linear-models/logistic_regression.ipynb), applied to $\text{score}(\text{spam}) - \text{score}(\text{ham})$, a weighted sum of the word counts: for two classes, naive Bayes is a linear classifier of the same form, whose weights come from counting instead of from minimising an error.

## The classifier

Everything above, in one class with scikit-learn's conventions: `fit` learns from the training messages and returns the object, what it learns is stored in attributes ending in `_`, and `predict_proba` and `predict` answer for new messages. Words that weren't in the training vocabulary carry no information about either class, and are skipped:

In [ ]:
class NaiveBayesSpamFilter:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, texts, labels):
        labels = pd.Series(list(labels))
        self.classes_ = ["ham", "spam"]
        self.log_prior_ = {label: np.log((labels == label).mean()) for label in self.classes_}
        counts = {label: Counter() for label in self.classes_}
        for text, label in zip(texts, labels):
            counts[label].update(words_in(text))
        self.vocabulary_ = set(counts["ham"]) | set(counts["spam"])
        self.log_likelihood_ = {}
        for label in self.classes_:
            denominator = sum(counts[label].values()) + self.alpha * len(self.vocabulary_)
            self.log_likelihood_[label] = {
                word: np.log((counts[label][word] + self.alpha) / denominator) for word in self.vocabulary_
            }
        return self

    def log_scores(self, text):
        known_words = [word for word in words_in(text) if word in self.vocabulary_]
        return {
            label: self.log_prior_[label] + sum(self.log_likelihood_[label][word] for word in known_words)
            for label in self.classes_
        }

    def predict_proba(self, texts):
        spam_probabilities = []
        for text in texts:
            scores = self.log_scores(text)
            spam_probabilities.append(1 / (1 + np.exp(scores["ham"] - scores["spam"])))
        return np.column_stack([1 - np.array(spam_probabilities), spam_probabilities])

    def predict(self, texts):
        return np.where(self.predict_proba(texts)[:, 1] > 0.5, "spam", "ham")

In [ ]:
spam_filter = NaiveBayesSpamFilter(alpha=1.0).fit(train["message"], train["label"])

examples = [
    "Congratulations! You have won a free prize. Call now to claim it",
    "Are we still meeting for lunch tomorrow?",
    "Free entry to the party tonight, just text me when you arrive",
]
pd.DataFrame({"P(spam)": spam_filter.predict_proba(examples)[:, 1].round(4), "prediction": spam_filter.predict(examples)}, index=examples)

,P(spam),prediction
Congratulations! You have won a free prize. Call now to claim it,1.0000,spam
Are we still meeting for lunch tomorrow?,0.0000,ham
"Free entry to the party tonight, just text me when you arrive",0.2858,ham


The first message is blatant spam and the second plain ham, and the filter is certain of both: 1.0000 and 0.0000, to four decimals. The third uses spam's favourite words, "free", "entry", "text", in a message between friends, and the filter hesitates: 0.29, ham. Notice how extreme the first two probabilities are. Multiplying dozens of word probabilities as if they were independent makes naive Bayes far more confident than it should be: its probabilities rank messages well, but they shouldn't be read literally.

On the test messages, with the measures of [Evaluating a classifier](../../evaluation/classification_metrics.ipynb); "spam" is the positive class:

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

test_predictions = spam_filter.predict(test["message"])
pd.DataFrame(confusion_matrix(test["label"], test_predictions, labels=["ham", "spam"]),
             index=["true ham", "true spam"], columns=["predicted ham", "predicted spam"])

,predicted ham,predicted spam
true ham,900,4
true spam,10,121


In [ ]:
{
    "accuracy": (test_predictions == test["label"]).mean(),
    "precision (spam)": precision_score(test["label"], test_predictions, pos_label="spam"),
    "recall (spam)": recall_score(test["label"], test_predictions, pos_label="spam"),
}

{'accuracy': np.float64(0.9864734299516909),
 'precision (spam)': 0.968,
 'recall (spam)': 0.9236641221374046}

Of the 1,035 test messages, 14 are misclassified: 4 ham messages sent to spam, and 10 spam messages let through. When the filter says spam it is right 97% of the time, and it catches 92% of the spam. For a filter, the four ham messages weigh more than the ten spam ones: a lost legitimate message costs more than a spam message deleted by hand. [The spam filter with scikit-learn](spam_filter_scikit_learn.ipynb) looks at that trade-off.

## The same with scikit-learn

scikit-learn splits the work in two: `CountVectorizer` turns each message into its word counts, with the same lower-casing and the same pattern for words, and `MultinomialNB` is the classifier above, smoothing included. Fitted on the same training messages, they should give the same probabilities:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

library_filter = make_pipeline(CountVectorizer(token_pattern=r"[a-z0-9£]+"), MultinomialNB(alpha=1.0))
library_filter.fit(train["message"], train["label"])
library_probabilities = library_filter.predict_proba(test["message"])[:, 1]
our_probabilities = spam_filter.predict_proba(test["message"])[:, 1]
{
    "same predictions": (library_filter.predict(test["message"]) == test_predictions).mean(),
    "largest difference in P(spam)": np.abs(library_probabilities - our_probabilities).max(),
}

{'same predictions': np.float64(1.0),
 'largest difference in P(spam)': np.float64(2.6756374893466273e-14)}

Every prediction is the same, and the probabilities differ by less than $10^{-13}$, the rounding error of floating-point arithmetic: a few lines of Python and scikit-learn's `MultinomialNB` are the same model.

[The spam filter with scikit-learn](spam_filter_scikit_learn.ipynb) starts from there: other ways of turning text into numbers, a comparison with another classifier, the costs of each kind of error, and what the filter gets wrong.